In [ ]:
import pandas as pd
import re
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)
df_tweet = pd.read_csv("raw_tweets_text.csv")

In [ ]:
from emoji import emoji_list

def extract_emoji(text, emoji_list=emoji_list):
    return [item["emoji"] for item in emoji_list(text)]


df_tweet["emojis"] = df_tweet["text"].parallel_map(extract_emoji)

In [ ]:
from emoji import demojize
# Transform emoticons to tokens
emoticon_dict = {":)": "emoji_smile", ":-)": "emoji_smile",
                 ":(": "emoji_sad", ":-(": "emoji_sad",
                 ":D": "emoji_laugh", ";)": "emoji_wink"}
HASHTAG = re.compile(r"#(\w+)")
REPEATED = re.compile(r"(.)\1{3,}")
# remove url, retweet, mentions, numbers
PATTERN_REMOVE = re.compile('|'.join([r"https?://\S+|www\.\S+", r"@\w+", r"\brt\b", r"\d+"]))
EMO_MAP = {re.escape(k): v for k, v in emoticon_dict.items()}
PATTERN_EMOTICON = re.compile("|".join(re.escape(k) for k in sorted(EMO_MAP.keys(), key=len, reverse=True)))


def clean_text(text, demojize=demojize, emoticon=PATTERN_EMOTICON, remove=PATTERN_REMOVE, hashtag=HASHTAG, repeated=REPEATED):
    text = demojize(text, delimiters=(" emoji_", " "))
    text = text.lower()
    text = emoticon.sub(lambda m: f" {EMO_MAP[m.group(0)]} ", text)
    text = remove.sub("", text)
    text = hashtag.sub(r"\1", text)
    text = repeated.sub(r"\1\1\1", text)
    return " ".join(text.split()).strip()


df_tweet["text_clean"] = df_tweet["text"].parallel_map(clean_text)

In [ ]:
from functools import cache
from contractions import fix
from string import punctuation
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

# Setup NLTK
tokenizer = TweetTokenizer(preserve_case=False)
sentence_breakers = set(punctuation) | {'but', 'however', 'although', 'though', 'even though', 'whereas'}
negation_words = {'not', 'no', 'never', 'neither', 'nor',
                  "can't", "cannot", "don't", "didn't", "doesn't", "isn't", "wasn't", "weren't", "ain't"}
stopwords_set = set(stopwords.words('english')) - negation_words
lemmatizer = WordNetLemmatizer()

NEG = 3  # Number of words after which negation effect ends


def get_wordnet_pos(tag, ADJ=wordnet.ADJ, VERB=wordnet.VERB, NOUN=wordnet.NOUN, ADV=wordnet.ADV):
    # Function to convert NLTK POS tags to WordNet POS tags
    if tag.startswith('J'):
        return ADJ
    elif tag.startswith('V'):
        return VERB
    elif tag.startswith('N'):
        return NOUN
    elif tag.startswith('R'):
        return ADV
    else:
        return NOUN  # Default

@cache
def lemmatize(word, tag, lemmatizer=lemmatizer, get_wordnet_pos=get_wordnet_pos):
    return lemmatizer.lemmatize(word, get_wordnet_pos(tag))

def text_process(text, fix=fix, tokenize=tokenizer.tokenize, pos_tag=pos_tag, sentence_breakers=sentence_breakers, lemmatize=lemmatize, negation_words=negation_words, NEG=NEG):
    text = fix(text)
    tokens = tokenize(text)
    tagged_tokens = pos_tag(tokens)
    final = []
    negation = 0
    for word, tag in tagged_tokens:
        if word in sentence_breakers or not word.isalnum():
            negation = 0
        elif word.startswith("emoji_"):
            final.append(word)
        else:
            lemma = lemmatize(word, tag)
            if word in negation_words:
                negation = NEG
                final.append(lemma)
            elif negation > 0:
                final.append(f"NOT_{lemma}")
                negation -= 1
            else:
                final.append(lemma)
    return final


df_tweet["final_tokens"] = df_tweet["text_clean"].parallel_map(text_process)

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split


emoji_db = pd.read_csv("Emoji_Sentiment.csv", encoding="utf-8")  # Dataset with emoji sentiment scores
emoji_db["score"] = (emoji_db["Positive"] - emoji_db["Negative"]) / (emoji_db["Positive"] + emoji_db["Negative"] + emoji_db["Neutral"])  # Calculate sentiment score as normalized difference
emoji_lookup = dict(zip(emoji_db["Emoji"], emoji_db["score"]))

sia = SentimentIntensityAnalyzer()  # Inizialize VADER sentiment analyzer


def get_score_emoji(emoji_list, emoji_lookup=emoji_lookup):
    if not emoji_list:
        return 0.0
    score = [emoji_lookup.get(e, 0.0) for e in emoji_list]
    return sum(score) / len(score)


def vader_label(text_score, margin=0.05):
    # Function to convert VADER compound score to labels
    if text_score > margin:
        return "Positive"
    elif text_score < -margin:
        return "Negative"
    else:
        return "Neutral"


def sentiment(text, vader_label=vader_label, sia=sia):
    return vader_label(sia.polarity_scores(str(text))['compound'])


df_tweet["sentiment"] = df_tweet["text_clean"].parallel_map(sentiment)


# Calculate emoji score using another dataset
df_tweet["emoji_score"] = df_tweet["emojis"].parallel_map(get_score_emoji)

print("Distribuzione sentiment:", df_tweet["sentiment"].value_counts())

# Prepare Features & Target
X_text = df_tweet["final_tokens"].parallel_map(" ".join)  # Join tokens into string
X_extra = df_tweet[["emoji_score"]].values  # Emoji score as a numeric feature
y = df_tweet["sentiment"]  # Target sentiment

# Train/test split
X_train_text, X_test_text, X_train_extra, X_test_extra, y_train, y_test = train_test_split(X_text, X_extra, y, test_size=0.2, random_state=42, stratify=y)

# TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 3), min_df=5, max_df=0.8, sublinear_tf=True, max_features=50000)  # Consider unigrams to n-grams, ignore rare terms (min_df), ignore very common terms (max_df) and
# apply logarithmic scaling to term frequency
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

scaler = StandardScaler(with_mean=False)

X_train_extra_scaled = scaler.fit_transform(X_train_extra)
X_test_extra_scaled = scaler.transform(X_test_extra)

# Emoji score shift for Naive Bayes
X_train_extra_nb = X_train_extra + 1  # shift [-1,1]->[0,2]
X_test_extra_nb = X_test_extra + 1

X_train_final_nb = hstack([X_train_tfidf, X_train_extra_nb])
X_test_final_nb = hstack([X_test_tfidf, X_test_extra_nb])

# With weighting emoji score
X_train_extra_scaled_weighted = X_train_extra_scaled * 0.8
X_test_extra_scaled_weighted = X_test_extra_scaled * 0.8

# Union of TF-IDF features and emoji_score
X_train_final_lr = hstack([X_train_tfidf, X_train_extra_scaled_weighted])
X_test_final_lr = hstack([X_test_tfidf, X_test_extra_scaled_weighted])


df_tweet.sample(10)

In [ ]:
# Test using MultinomialNB - Confusion Matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# Train Multinomial Naive Bayes classifier with Laplace Smoothing (alpha=1)
model = MultinomialNB(alpha=1)
model.fit(X_train_final_nb, y_train)
y_predMultinomial = model.predict(X_test_final_nb)  # Predict and test

print(f"Global Accuracy with MultinomialNB: {accuracy_score(y_test, y_predMultinomial):.2f}\n")
print(classification_report(y_test, y_predMultinomial))
cm = confusion_matrix(y_test, y_predMultinomial, labels=["Positive", "Neutral", "Negative"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Positive", "Neutral", "Negative"], yticklabels=["Positive", "Neutral", "Negative"])
plt.title("Confusion Matrix with MultinomialNB")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Test using ComplementNB
from sklearn.naive_bayes import ComplementNB


model2 = ComplementNB(alpha=1)  # Using Laplace smoothing
model2.fit(X_train_final_nb, y_train)
y_predComplement = model2.predict(X_test_final_nb)
print(f"Global Accuracy with ComplementNB: {accuracy_score(y_test, y_predComplement):.2f}\n")
print(classification_report(y_test, y_predComplement))

cmw = confusion_matrix(y_test, y_predComplement, labels=["Positive", "Neutral", "Negative"])
sns.heatmap(cmw, annot=True, fmt="d", cmap="Blues", xticklabels=["Positive", "Neutral", "Negative"], yticklabels=["Positive", "Neutral", "Negative"])
plt.title("Confusion Matrix with ComplementNB")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Test using LogisticRegression
from sklearn.linear_model import LogisticRegression

modelLogistic = LogisticRegression(max_iter=1000, solver='saga', class_weight='balanced', C=2, verbose=1)
modelLogistic.fit(X_train_final_lr, y_train)

y_predRegression = modelLogistic.predict(X_test_final_lr)

print(f"Global Accuracy with LogisticRegression: {accuracy_score(y_test, y_predRegression):.2f}\n")
print(classification_report(y_test, y_predRegression))

cmr = confusion_matrix(y_test, y_predRegression, labels=["Positive", "Neutral", "Negative"])
sns.heatmap(cmr, annot=True, fmt="d", cmap="Blues", xticklabels=["Positive", "Neutral", "Negative"], yticklabels=["Positive", "Neutral", "Negative"])
plt.title("Confusion Matrix with LogisticRegression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()